# 07 - Model Training

## Objective

This notebook prepares leakage-safe modeling datasets, establishes baselines, performs deep two-stage hyperparameter tuning, and selects a candidate model using five expanding chronological folds.

Logistic Regression, Random Forest, and XGBoost are compared with identical standard-Python preprocessing, chronological samples, class-imbalance alternatives, probability metrics, operational metrics, and computational constraints. Decision thresholds are optimized consistently from out-of-fold predictions so delayed-flight Precision, Recall, F1, and F2 are comparable across imbalance strategies.

The notebook persists the complete tuning evidence, preprocessing contract, selected algorithm, hyperparameters, imbalance strategy, and cross-validated operating threshold. The November–December holdout outcomes remain untouched during candidate selection.


#### Load project configuration


In [0]:
import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

# Load the project configuration

from config import project_config as cfg

print("Project configuration loaded successfully.")


#### Load and validate the feature dataset

The model-training process begins by loading the managed `flights_features` Delta table produced by the Feature Engineering notebook.

Before splitting or modelling, the dataset is validated to confirm that:

- The required Unity Catalog table exists
- The target variable is available
- The flight date is stored as a valid date
- All required schedule-time predictors are present
- The dataset contains records suitable for chronological splitting


In [0]:
from __future__ import annotations

import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

from config import project_config as cfg
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T



FEATURE_TABLE = cfg.FEATURES_TABLE
TARGET_COLUMN = cfg.TARGET_COLUMN
DATE_COLUMN = cfg.FLIGHT_DATE_COLUMN


def require_table(table_name: str) -> None:
    """Raise an error when a required Unity Catalog table is unavailable."""
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table '{table_name}' was not found. "
            "Run the feature-engineering notebook (06) before continuing."
        )


require_table(FEATURE_TABLE)

df_features: DataFrame = spark.table(FEATURE_TABLE)

required_columns = cfg.MODEL_TRAINING_REQUIRED_COLUMNS

missing_columns = sorted(required_columns - set(df_features.columns))

if missing_columns:
    raise ValueError(
        "Model-training validation failed. "
        f"Missing required columns: {missing_columns}"
    )

feature_row_count = df_features.count()
feature_column_count = len(df_features.columns)

date_type = df_features.schema[DATE_COLUMN].dataType

if not isinstance(date_type, T.DateType):
    raise TypeError(
        f"{DATE_COLUMN} must be a Spark date column, "
        f"but found {date_type.simpleString()}."
    )

print("Feature dataset loaded and validated successfully.")
print(f"Source table: {FEATURE_TABLE}")
print(f"Total records: {feature_row_count:,}")
print(f"Total columns: {feature_column_count}")
print(f"Prediction target: {TARGET_COLUMN}")
print(f"Date column type: {date_type.simpleString()}")

#### Date range and target distribution

Before defining the chronological training, validation, and test periods, the feature dataset is examined to confirm its available date range and the distribution of the binary target variable.

This review supports two important modelling decisions:

- Selecting non-overlapping chronological split periods
- Assessing whether the delayed and on-time classes are imbalanced

No records are modified during this analysis.


In [0]:
dataset_profile = (
    df_features
    .select(
        F.min("FL_DATE").alias("MIN_FL_DATE"),
        F.max("FL_DATE").alias("MAX_FL_DATE"),
        F.count("*").alias("TOTAL_RECORDS"),
        F.sum(
            F.when(F.col("ARR_DEL15") == 0, 1).otherwise(0)
        ).alias("ON_TIME_RECORDS"),
        F.sum(
            F.when(F.col("ARR_DEL15") == 1, 1).otherwise(0)
        ).alias("DELAYED_RECORDS"),
    )
    .withColumn(
        "ON_TIME_PERCENTAGE",
        F.round(
            F.col("ON_TIME_RECORDS") / F.col("TOTAL_RECORDS") * 100,
            4,
        ),
    )
    .withColumn(
        "DELAYED_PERCENTAGE",
        F.round(
            F.col("DELAYED_RECORDS") / F.col("TOTAL_RECORDS") * 100,
            4,
        ),
    )
)

display(dataset_profile)

#### Create chronological train, validation, and test splits

The dataset is divided chronologically rather than randomly because the model is intended to predict future flight-delay risk from historical observations.

The split periods are defined as follows:

- **Training period:** January 1, 2025 to August 31, 2025
- **Validation period:** September 1, 2025 to October 31, 2025
- **Test period:** November 1, 2025 to December 31, 2025

This design ensures that later flight outcomes are not used to train models evaluated on earlier periods. The validation dataset will support model and hyperparameter selection, while the test dataset will remain untouched until final evaluation.


In [0]:
TRAIN_END_DATE = cfg.TRAIN_END_DATE
VALIDATION_START_DATE = cfg.VALIDATION_START_DATE
VALIDATION_END_DATE = cfg.VALIDATION_END_DATE
TEST_START_DATE = cfg.TEST_START_DATE

df_train = df_features.filter(
    F.col("FL_DATE") <= F.to_date(F.lit(TRAIN_END_DATE))
)

df_validation = df_features.filter(
    (F.col("FL_DATE") >= F.to_date(F.lit(VALIDATION_START_DATE)))
    & (F.col("FL_DATE") <= F.to_date(F.lit(VALIDATION_END_DATE)))
)

df_test = df_features.filter(
    F.col("FL_DATE") >= F.to_date(F.lit(TEST_START_DATE))
)

split_summary = (
    df_train.select(
        F.lit("TRAIN").alias("DATASET"),
        F.min("FL_DATE").alias("MIN_DATE"),
        F.max("FL_DATE").alias("MAX_DATE"),
        F.count("*").alias("TOTAL_RECORDS"),
        F.avg(F.col("ARR_DEL15").cast("double")).alias("DELAY_RATE"),
    )
    .unionByName(
        df_validation.select(
            F.lit("VALIDATION").alias("DATASET"),
            F.min("FL_DATE").alias("MIN_DATE"),
            F.max("FL_DATE").alias("MAX_DATE"),
            F.count("*").alias("TOTAL_RECORDS"),
            F.avg(F.col("ARR_DEL15").cast("double")).alias("DELAY_RATE"),
        )
    )
    .unionByName(
        df_test.select(
            F.lit("TEST").alias("DATASET"),
            F.min("FL_DATE").alias("MIN_DATE"),
            F.max("FL_DATE").alias("MAX_DATE"),
            F.count("*").alias("TOTAL_RECORDS"),
            F.avg(F.col("ARR_DEL15").cast("double")).alias("DELAY_RATE"),
        )
    )
    .withColumn(
        "DELAY_PERCENTAGE",
        F.round(F.col("DELAY_RATE") * 100, 4),
    )
    .drop("DELAY_RATE")
)

display(split_summary)

#### Split validation summary

The chronological split produced three non-overlapping datasets whose combined record count matches the complete feature dataset.

The target distribution varies across the periods:

- The training period has a delay rate of approximately 22.91%.
- The validation period has a lower delay rate of approximately 18.55%.
- The test period has a higher delay rate of approximately 23.71%.

This variation reflects temporal changes in airline operations and confirms the importance of evaluating the model on future periods rather than using a random split.


#### Engineer Strictly Leakage-Safe Historical Features

Historical delay-rate features summarize prior airline, airport, and route performance. For every training date, both the entity history and its smoothing prior are calculated from **strictly earlier dates only**. Outcomes from the current date and all later dates are excluded.

The initial value `0.5` is used only when the dataset contains no earlier observations. Once earlier flights exist, the cumulative delay rate through the preceding date becomes the date-specific smoothing prior. This removes the former January–August global fallback that allowed early flights to indirectly receive information from later months.


In [0]:
from pyspark.sql.window import Window

SMOOTHING_STRENGTH = float(cfg.HISTORICAL_SMOOTHING_STRENGTH)
INITIAL_DELAY_PRIOR = float(cfg.HISTORICAL_INITIAL_PRIOR_RATE)

# Causal global prior for each date, calculated from earlier dates only.
global_daily_stats = (
    df_train
    .groupBy("FL_DATE")
    .agg(
        F.count("*").alias("GLOBAL_DAILY_FLIGHTS"),
        F.sum(F.col(TARGET_COLUMN).cast("long")).alias(
            "GLOBAL_DAILY_DELAYS"
        ),
    )
)

global_history_window = (
    Window
    .orderBy(F.col("FL_DATE").cast("timestamp").cast("long"))
    .rowsBetween(Window.unboundedPreceding, -1)
)

date_specific_priors = (
    global_daily_stats
    .withColumn(
        "GLOBAL_PRIOR_FLIGHTS",
        F.sum("GLOBAL_DAILY_FLIGHTS").over(global_history_window),
    )
    .withColumn(
        "GLOBAL_PRIOR_DELAYS",
        F.sum("GLOBAL_DAILY_DELAYS").over(global_history_window),
    )
    .withColumn(
        "DATE_PRIOR_DELAY_RATE",
        F.when(
            F.col("GLOBAL_PRIOR_FLIGHTS").isNull()
            | (F.col("GLOBAL_PRIOR_FLIGHTS") == 0),
            F.lit(INITIAL_DELAY_PRIOR),
        ).otherwise(
            F.col("GLOBAL_PRIOR_DELAYS").cast("double")
            / F.col("GLOBAL_PRIOR_FLIGHTS").cast("double")
        ),
    )
    .select("FL_DATE", "DATE_PRIOR_DELAY_RATE")
)


def causal_entity_history(source_df, entity_columns, output_prefix):
    '''Create a smoothed entity rate using strictly earlier dates.'''
    daily_flights = f"{output_prefix}_DAILY_FLIGHTS"
    daily_delays = f"{output_prefix}_DAILY_DELAYS"
    prior_flights = f"{output_prefix}_PRIOR_FLIGHTS"
    prior_delays = f"{output_prefix}_PRIOR_DELAYS"
    rate_column = f"{output_prefix}_HIST_DELAY_RATE"

    daily = (
        source_df
        .groupBy(*entity_columns, "FL_DATE")
        .agg(
            F.count("*").alias(daily_flights),
            F.sum(F.col(TARGET_COLUMN).cast("long")).alias(daily_delays),
        )
    )
    history_window = (
        Window
        .partitionBy(*entity_columns)
        .orderBy(F.col("FL_DATE").cast("timestamp").cast("long"))
        .rowsBetween(Window.unboundedPreceding, -1)
    )

    return (
        daily
        .withColumn(prior_flights, F.sum(daily_flights).over(history_window))
        .withColumn(prior_delays, F.sum(daily_delays).over(history_window))
        .join(date_specific_priors, on="FL_DATE", how="left")
        .withColumn(
            rate_column,
            (
                F.coalesce(F.col(prior_delays).cast("double"), F.lit(0.0))
                + F.lit(SMOOTHING_STRENGTH)
                * F.col("DATE_PRIOR_DELAY_RATE")
            )
            /
            (
                F.coalesce(F.col(prior_flights).cast("double"), F.lit(0.0))
                + F.lit(SMOOTHING_STRENGTH)
            ),
        )
        .select(
            *entity_columns,
            "FL_DATE",
            prior_flights,
            prior_delays,
            rate_column,
        )
    )


print("Strictly causal date-specific smoothing priors created.")


#### Historical Airline Delay Rate

`AIRLINE_HIST_DELAY_RATE` uses the airline's outcomes from earlier dates and the causal date-specific prior. The current date and later months cannot influence the value.


In [0]:
airline_history_features = causal_entity_history(
    df_train,
    ["OP_UNIQUE_CARRIER"],
    "AIRLINE",
)

df_train_hist = (
    df_train
    .join(
        airline_history_features,
        on=["OP_UNIQUE_CARRIER", "FL_DATE"],
        how="left",
    )
    .join(date_specific_priors, on="FL_DATE", how="left")
    .withColumn(
        "AIRLINE_HIST_DELAY_RATE",
        F.coalesce(
            F.col("AIRLINE_HIST_DELAY_RATE"),
            F.col("DATE_PRIOR_DELAY_RATE"),
            F.lit(INITIAL_DELAY_PRIOR),
        ),
    )
    .drop("DATE_PRIOR_DELAY_RATE")
)

print(f"Training rows after causal airline join: {df_train_hist.count():,}")


#### Historical Origin-Airport Delay Rate

`ORIGIN_HIST_DELAY_RATE` uses only flights from earlier dates at the same origin, smoothed toward the cumulative delay rate available before the current date.


In [0]:
origin_history_features = causal_entity_history(
    df_train,
    ["ORIGIN"],
    "ORIGIN",
)

df_train_hist = (
    df_train_hist
    .join(origin_history_features, on=["ORIGIN", "FL_DATE"], how="left")
    .join(date_specific_priors, on="FL_DATE", how="left")
    .withColumn(
        "ORIGIN_HIST_DELAY_RATE",
        F.coalesce(
            F.col("ORIGIN_HIST_DELAY_RATE"),
            F.col("DATE_PRIOR_DELAY_RATE"),
            F.lit(INITIAL_DELAY_PRIOR),
        ),
    )
    .drop("DATE_PRIOR_DELAY_RATE")
)


#### Historical Destination-Airport Delay Rate

`DEST_HIST_DELAY_RATE` uses only flights from earlier dates with the same destination, with the same strictly causal smoothing rule.


In [0]:
dest_history_features = causal_entity_history(
    df_train,
    ["DEST"],
    "DEST",
)

df_train_hist = (
    df_train_hist
    .join(dest_history_features, on=["DEST", "FL_DATE"], how="left")
    .join(date_specific_priors, on="FL_DATE", how="left")
    .withColumn(
        "DEST_HIST_DELAY_RATE",
        F.coalesce(
            F.col("DEST_HIST_DELAY_RATE"),
            F.col("DATE_PRIOR_DELAY_RATE"),
            F.lit(INITIAL_DELAY_PRIOR),
        ),
    )
    .drop("DATE_PRIOR_DELAY_RATE")
)


#### Historical Route Delay Rate

`ROUTE_HIST_DELAY_RATE` uses only earlier dates on the same origin–destination route. Sparse routes are smoothed toward the causal prior available before that flight date.


In [0]:
route_history_features = causal_entity_history(
    df_train,
    ["ORIGIN", "DEST"],
    "ROUTE",
)

df_train_hist = (
    df_train_hist
    .join(
        route_history_features,
        on=["ORIGIN", "DEST", "FL_DATE"],
        how="left",
    )
    .join(date_specific_priors, on="FL_DATE", how="left")
    .withColumn(
        "ROUTE_HIST_DELAY_RATE",
        F.coalesce(
            F.col("ROUTE_HIST_DELAY_RATE"),
            F.col("DATE_PRIOR_DELAY_RATE"),
            F.lit(INITIAL_DELAY_PRIOR),
        ),
    )
    .drop("DATE_PRIOR_DELAY_RATE")
)

print("All training historical rates use strictly earlier dates.")


#### Apply Frozen Training History to Later Periods

Historical-rate mappings for a validation period are learned exclusively from observations available before that period begins. The September–October validation and November–December holdout datasets therefore use mappings learned from January–August only.

The same helper is reused inside cross-validation: each validation month receives mappings learned only from its corresponding earlier training window. Validation targets never participate in their own feature construction. Unseen airlines, airports, or routes fall back to the global delay rate from the applicable training window.


In [0]:
HISTORY_ENTITY_SPECS = [
    (["OP_UNIQUE_CARRIER"], "AIRLINE"),
    (["ORIGIN"], "ORIGIN"),
    (["DEST"], "DEST"),
    (["ORIGIN", "DEST"], "ROUTE"),
]


def build_frozen_history_maps(training_dataframe: DataFrame):
    '''Learn smoothed historical mappings from one training window only.'''
    global_rate = (
        training_dataframe
        .select(F.avg(F.col(TARGET_COLUMN).cast("double")).alias("RATE"))
        .first()["RATE"]
    )
    if global_rate is None:
        raise ValueError("Cannot build historical mappings from empty training data.")

    history_maps = {}
    for entity_columns, prefix in HISTORY_ENTITY_SPECS:
        flights_column = f"{prefix}_TRAIN_FLIGHTS"
        delays_column = f"{prefix}_TRAIN_DELAYS"
        rate_column = f"{prefix}_HIST_DELAY_RATE"

        history_maps[prefix] = (
            training_dataframe
            .groupBy(*entity_columns)
            .agg(
                F.count("*").alias(flights_column),
                F.sum(F.col(TARGET_COLUMN).cast("long")).alias(delays_column),
            )
            .withColumn(
                rate_column,
                (
                    F.col(delays_column).cast("double")
                    + F.lit(SMOOTHING_STRENGTH * global_rate)
                )
                /
                (
                    F.col(flights_column).cast("double")
                    + F.lit(SMOOTHING_STRENGTH)
                ),
            )
            .select(*entity_columns, rate_column)
        )

    return float(global_rate), history_maps


def attach_frozen_history(
    dataset: DataFrame,
    global_rate: float,
    history_maps: dict[str, DataFrame],
) -> DataFrame:
    '''Attach mappings learned from an earlier training window.'''
    result = dataset
    for entity_columns, prefix in HISTORY_ENTITY_SPECS:
        result = result.join(
            history_maps[prefix],
            on=entity_columns,
            how="left",
        )

    return result.fillna(
        {
            f"{prefix}_HIST_DELAY_RATE": global_rate
            for _, prefix in HISTORY_ENTITY_SPECS
        }
    )


global_training_delay_rate, development_history_maps = (
    build_frozen_history_maps(df_train)
)
df_validation_hist = attach_frozen_history(
    df_validation,
    global_training_delay_rate,
    development_history_maps,
)
df_test_hist = attach_frozen_history(
    df_test,
    global_training_delay_rate,
    development_history_maps,
)

print(f"Validation rows after historical joins: {df_validation_hist.count():,}")
print(f"Test rows after historical joins: {df_test_hist.count():,}")

display(
    df_validation_hist
    .select(
        "FL_DATE",
        "OP_UNIQUE_CARRIER",
        "ORIGIN",
        "DEST",
        *cfg.MODEL_HISTORICAL_RATE_COLUMNS,
        TARGET_COLUMN,
    )
    .limit(20)
)


#### Validate historical performance features

The historical feature datasets are validated before categorical encoding and model training.

This check confirms that:

- Record counts remain unchanged after historical-feature joins
- No historical delay-rate features contain missing values
- All historical rates fall within the valid probability range of 0 to 1
- Training, validation, and test datasets contain the same historical feature columns


In [0]:
HISTORICAL_RATE_COLUMNS = list(cfg.MODEL_HISTORICAL_RATE_COLUMNS)

historical_validation_rows = []

for dataset_name, dataset in [
    ("TRAIN", df_train_hist),
    ("VALIDATION", df_validation_hist),
    ("TEST", df_test_hist),
]:
    summary_row = (
        dataset
        .select(
            F.lit(dataset_name).alias("DATASET"),
            F.count("*").alias("TOTAL_RECORDS"),
            *[
                F.sum(
                    F.when(F.col(column_name).isNull(), 1).otherwise(0)
                ).alias(f"{column_name}_NULLS")
                for column_name in HISTORICAL_RATE_COLUMNS
            ],
            *[
                F.sum(
                    F.when(
                        (F.col(column_name) < 0)
                        | (F.col(column_name) > 1),
                        1,
                    ).otherwise(0)
                ).alias(f"{column_name}_OUT_OF_RANGE")
                for column_name in HISTORICAL_RATE_COLUMNS
            ],
        )
    )

    historical_validation_rows.append(summary_row)

historical_validation_summary = historical_validation_rows[0]

for summary_row in historical_validation_rows[1:]:
    historical_validation_summary = (
        historical_validation_summary.unionByName(summary_row)
    )

display(historical_validation_summary)


#### Prepare Raw Predictor Columns for Standard Python

The leakage-safe historical training, validation, and test frames retain their original categorical and numerical predictor columns. Spark is used only to prepare, filter, and sample these large datasets.

Categorical encoding is deliberately deferred until after each chronological training fold has been sampled. A scikit-learn `ColumnTransformer` is fitted only on that fold's training observations and then applied to its later validation month. This keeps category learning inside the training boundary and avoids preprocessing leakage.


In [0]:
import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

from utils.model_training import prepare_hist_modeling_frame

CATEGORICAL_COLUMNS = list(cfg.MODEL_CATEGORICAL_COLUMNS)
NUMERICAL_COLUMNS = list(cfg.MODEL_NUMERICAL_COLUMNS)
MODEL_INPUT_COLUMNS = list(cfg.MODEL_INPUT_COLUMNS)

df_train_hist = prepare_hist_modeling_frame(df_train_hist)
df_validation_hist = prepare_hist_modeling_frame(df_validation_hist)
df_test_hist = prepare_hist_modeling_frame(df_test_hist)

df_train_prepared = df_train_hist.select(
    "FL_DATE", *MODEL_INPUT_COLUMNS, TARGET_COLUMN
)
df_validation_prepared = df_validation_hist.select(
    "FL_DATE", *MODEL_INPUT_COLUMNS, TARGET_COLUMN
)
df_test_prepared = df_test_hist.select(
    "FL_DATE", *MODEL_INPUT_COLUMNS, TARGET_COLUMN
)

print("Raw predictor datasets prepared for scikit-learn preprocessing.")
print(f"Categorical predictors: {len(CATEGORICAL_COLUMNS)}")
print(f"Numerical predictors: {len(NUMERICAL_COLUMNS)}")
print(f"Total predictors: {len(MODEL_INPUT_COLUMNS)}")


#### Validate the Raw Model Datasets

The raw predictor schema is validated before sampling. Each dataset must contain the flight date, target, and exactly the predictor columns expected by the standard-Python preprocessing pipeline.

No categorical mappings or numerical imputation values are learned at this stage.


In [0]:
required_model_columns = {
    "FL_DATE", TARGET_COLUMN, *MODEL_INPUT_COLUMNS
}

for dataframe_name, dataframe in {
    "df_train_prepared": df_train_prepared,
    "df_validation_prepared": df_validation_prepared,
    "df_test_prepared": df_test_prepared,
}.items():
    missing_columns = sorted(
        required_model_columns - set(dataframe.columns)
    )
    if missing_columns:
        raise ValueError(
            f"{dataframe_name} is missing columns: {missing_columns}"
        )
    if dataframe.limit(1).count() == 0:
        raise ValueError(f"{dataframe_name} contains no rows.")
    print(f"{dataframe_name} schema validated successfully.")


#### Standard-Python Preprocessing Boundary

After bounded sampling, raw Spark rows are collected into pandas. Scikit-learn then performs median numerical imputation, most-frequent categorical imputation, sparse one-hot encoding with unknown-category handling, and sparse-safe numerical scaling.

Every preprocessing transformer is fitted on training rows only. Logistic Regression, Random Forest, and XGBoost receive the same transformed sparse matrices within each fold.


In [0]:
print("Standard-Python preprocessing inputs are ready.")
print(f"Training rows: {df_train_prepared.count():,}")
print(f"Validation rows: {df_validation_prepared.count():,}")
print(f"Test rows: {df_test_prepared.count():,}")

display(
    df_train_prepared.select(
        "FL_DATE",
        *MODEL_INPUT_COLUMNS,
        TARGET_COLUMN,
    ).limit(10)
)


## Standard-Python Candidate Modeling

Logistic Regression, Random Forest, and XGBoost are implemented with standard Python libraries. Spark is limited to loading Delta tables, constructing leakage-safe historical features, enforcing chronological boundaries, and creating bounded samples before collection.

Every algorithm receives the same SciPy sparse matrices, five chronological folds, validation observations, imbalance alternatives, metrics, threshold-selection rule, and final-selection policy.


### Install Standard-Python Modeling Dependencies

XGBoost is pinned for reproducibility. The installed NumPy, pandas, SciPy, and scikit-learn versions are recorded later in the persisted candidate metadata so the training environment can be reconstructed.


In [0]:
%pip install --quiet xgboost==2.1.4 scipy scikit-learn


In [0]:
try:
    import ast
    import time
    import warnings

    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import scipy
    import sklearn
    import xgboost

    from scipy.sparse import csr_matrix
    from sklearn.calibration import calibration_curve
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        accuracy_score,
        average_precision_score,
        brier_score_loss,
        confusion_matrix,
        precision_recall_fscore_support,
        roc_auc_score,
    )
    from sklearn.model_selection import ParameterSampler, StratifiedShuffleSplit
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from xgboost import XGBClassifier
except ImportError as error:
    raise ImportError(
        "Notebook 07 requires NumPy, pandas, SciPy, scikit-learn, and "
        "XGBoost. Run the dependency cell, restart Python once, and "
        "rerun the notebook from the beginning."
    ) from error

RANDOM_SEED = cfg.RANDOM_SEED
LOCAL_TRAIN_MAX_ROWS = 75_000
LOCAL_VALIDATION_MAX_ROWS = 20_000
BROAD_TRAIN_MAX_ROWS = 30_000
BROAD_VALIDATION_MAX_ROWS = 10_000
F2_BETA = 2.0
THRESHOLD_GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)

LIBRARY_VERSIONS = {
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
    "xgboost": xgboost.__version__,
}

print("Standard-Python modeling libraries loaded successfully.")
print(f"Library versions: {LIBRARY_VERSIONS}")
print(f"Confirmation training rows per fold: {LOCAL_TRAIN_MAX_ROWS:,}")
print(f"Confirmation validation rows per fold: {LOCAL_VALIDATION_MAX_ROWS:,}")
print(f"Broad-screening training rows per fold: {BROAD_TRAIN_MAX_ROWS:,}")
print(f"Broad-screening validation rows per fold: {BROAD_VALIDATION_MAX_ROWS:,}")
print(f"Cross-validated threshold candidates: {len(THRESHOLD_GRID)}")


## Reproducible Standard-Python Preprocessing and Evaluation Utilities

Raw chronological samples are transformed by a fresh scikit-learn `ColumnTransformer` fitted only on the corresponding training fold. Rare categorical levels are grouped during one-hot encoding to reduce dimensionality without using validation outcomes.

Tuning uses two computational tiers. Broad screening evaluates many configurations on reproducible stratified samples of 30,000 training and 10,000 validation rows per fold. The strongest broad configurations and their local refinements are then evaluated again on common confirmation samples of 75,000 training and 20,000 validation rows per fold. Only confirmation results can enter final model selection.

Model probabilities are evaluated with ROC-AUC, PR-AUC, Brier Score, Top-10% Recall, and Top-10% Lift. Every confirmed configuration receives the same out-of-fold threshold search, maximizing delayed-flight F2 while still penalizing false alerts.


In [0]:
def bounded_uniform_sample(dataframe, maximum_rows, *, seed):
    '''Return a seeded uniform sample with at most maximum_rows.'''
    row_count = dataframe.count()
    if row_count == 0:
        raise ValueError("Cannot sample an empty Spark DataFrame.")
    if row_count <= maximum_rows:
        return dataframe

    sampling_fraction = min(1.0, (maximum_rows * 1.10) / row_count)
    return (
        dataframe
        .sample(
            withReplacement=False,
            fraction=sampling_fraction,
            seed=seed,
        )
        .limit(maximum_rows)
    )


def spark_sample_to_pandas(dataframe):
    '''Collect one bounded raw Spark sample into pandas.'''
    local_frame = dataframe.select(
        *MODEL_INPUT_COLUMNS,
        TARGET_COLUMN,
    ).toPandas()
    if local_frame.empty:
        raise ValueError("The local modeling sample contains zero rows.")

    for column_name in CATEGORICAL_COLUMNS:
        values = local_frame[column_name].astype("object")
        local_frame[column_name] = values.where(pd.notna(values), np.nan)
    for column_name in NUMERICAL_COLUMNS:
        local_frame[column_name] = pd.to_numeric(
            local_frame[column_name],
            errors="coerce",
        )
    return local_frame


def build_sklearn_preprocessor():
    '''Build a fresh sparse transformer for one training boundary.'''
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="infrequent_if_exist",
                    min_frequency=20,
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )
    numerical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler(with_mean=False)),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("categorical", categorical_pipeline, CATEGORICAL_COLUMNS),
            ("numerical", numerical_pipeline, NUMERICAL_COLUMNS),
        ],
        sparse_threshold=1.0,
    )


def prepare_training_validation_matrices(
    training_dataframe,
    validation_dataframe,
):
    '''Fit preprocessing on training and transform later validation.'''
    training_local = spark_sample_to_pandas(training_dataframe)
    validation_local = spark_sample_to_pandas(validation_dataframe)

    preprocessor = build_sklearn_preprocessor()
    X_training = csr_matrix(
        preprocessor.fit_transform(training_local[MODEL_INPUT_COLUMNS]),
        dtype=np.float32,
    )
    X_validation = csr_matrix(
        preprocessor.transform(validation_local[MODEL_INPUT_COLUMNS]),
        dtype=np.float32,
    )
    y_training = training_local[TARGET_COLUMN].to_numpy(dtype=np.int8)
    y_validation = validation_local[TARGET_COLUMN].to_numpy(dtype=np.int8)

    return (
        preprocessor,
        X_training,
        y_training,
        X_validation,
        y_validation,
    )


def threshold_classification_metrics(labels, probabilities, threshold):
    '''Calculate threshold-dependent overall and delayed-class metrics.'''
    predictions = (probabilities >= threshold).astype(np.int8)

    weighted_precision, weighted_recall, weighted_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted",
            zero_division=0,
        )
    )
    delay_precision, delay_recall, delay_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="binary",
            pos_label=1,
            zero_division=0,
        )
    )
    beta_squared = F2_BETA**2
    delay_f2_denominator = (
        beta_squared * delay_precision + delay_recall
    )
    delay_f2 = (
        (1.0 + beta_squared) * delay_precision * delay_recall
        / delay_f2_denominator
        if delay_f2_denominator > 0
        else 0.0
    )

    return {
        "ACCURACY": float(accuracy_score(labels, predictions)),
        "PRECISION": float(weighted_precision),
        "RECALL": float(weighted_recall),
        "F1_SCORE": float(weighted_f1),
        "DELAY_PRECISION": float(delay_precision),
        "DELAY_RECALL": float(delay_recall),
        "DELAY_F1": float(delay_f1),
        "DELAY_F2": float(delay_f2),
    }


def probability_ranking_metrics(labels, probabilities):
    '''Calculate threshold-independent ranking and calibration metrics.'''
    top_count = max(1, int(np.ceil(len(labels) * 0.10)))
    top_indices = np.argsort(probabilities)[-top_count:]
    total_delays = int(np.sum(labels == 1))
    top_delays = int(np.sum(labels[top_indices] == 1))
    overall_delay_rate = float(np.mean(labels))
    top_delay_rate = float(np.mean(labels[top_indices]))

    return {
        "ROC_AUC": float(roc_auc_score(labels, probabilities)),
        "PR_AUC": float(average_precision_score(labels, probabilities)),
        "BRIER_SCORE": float(brier_score_loss(labels, probabilities)),
        "TOP_10_RECALL": float(
            top_delays / total_delays if total_delays else 0.0
        ),
        "TOP_10_LIFT": float(
            top_delay_rate / overall_delay_rate
            if overall_delay_rate
            else 0.0
        ),
    }


def evaluate_probabilities(labels, probabilities, *, threshold):
    '''Return the complete metric set at one decision threshold.'''
    return {
        **threshold_classification_metrics(
            labels,
            probabilities,
            threshold,
        ),
        **probability_ranking_metrics(labels, probabilities),
    }


def select_operating_threshold(labels, probabilities):
    '''Select the common-CV threshold that maximizes delayed-flight F2.'''
    candidates = []
    for threshold in THRESHOLD_GRID:
        metrics = threshold_classification_metrics(
            labels,
            probabilities,
            float(threshold),
        )
        candidates.append(
            (
                metrics["DELAY_F2"],
                metrics["DELAY_F1"],
                metrics["DELAY_RECALL"],
                metrics["DELAY_PRECISION"],
                -abs(float(threshold) - 0.5),
                float(threshold),
            )
        )
    return max(candidates)[-1]


def evaluate_python_classifier(
    model,
    X_validation,
    y_validation,
    *,
    threshold=0.5,
):
    probabilities = model.predict_proba(X_validation)[:, 1]
    return evaluate_probabilities(
        y_validation,
        probabilities,
        threshold=threshold,
    )


def positive_class_weight(labels):
    '''Return the negative-to-positive ratio for XGBoost.'''
    positive_count = int(np.sum(labels == 1))
    negative_count = int(np.sum(labels == 0))
    if positive_count == 0 or negative_count == 0:
        raise ValueError(
            "Both target classes must be present in every training fold."
        )
    return negative_count / positive_count



def stratified_local_sample(matrix, labels, maximum_rows, *, seed):
    '''Return a reproducible class-stratified local matrix sample.'''
    if len(labels) <= maximum_rows:
        return matrix, labels

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        train_size=maximum_rows,
        random_state=seed,
    )
    selected_indices, _ = next(
        splitter.split(np.zeros((len(labels), 1)), labels)
    )
    return matrix[selected_indices], labels[selected_indices]


def build_reduced_cv_folds(
    complete_folds,
    *,
    training_maximum,
    validation_maximum,
    seed_offset,
):
    '''Create smaller shared folds for broad hyperparameter screening.'''
    reduced_folds = []
    for fold in complete_folds:
        X_train, y_train = stratified_local_sample(
            fold["X_train"],
            fold["y_train"],
            training_maximum,
            seed=RANDOM_SEED + seed_offset + fold["fold"] * 10,
        )
        X_validation, y_validation = stratified_local_sample(
            fold["X_validation"],
            fold["y_validation"],
            validation_maximum,
            seed=RANDOM_SEED + seed_offset + fold["fold"] * 10 + 1,
        )
        reduced_folds.append(
            {
                **fold,
                "X_train": X_train,
                "y_train": y_train,
                "X_validation": X_validation,
                "y_validation": y_validation,
                "scale_pos_weight": positive_class_weight(y_train),
            }
        )
    return reduced_folds


def confusion_matrix_table(y_true, y_predicted):
    matrix = confusion_matrix(y_true, y_predicted, labels=[0, 1])
    return pd.DataFrame(
        matrix,
        index=["Actual On Time (0)", "Actual Delayed (1)"],
        columns=["Predicted On Time (0)", "Predicted Delayed (1)"],
    )


def confusion_count_table(y_true, y_predicted):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_predicted,
        labels=[0, 1],
    ).ravel()
    meanings = {
        "TN": "Actual on-time flight predicted as on time",
        "FP": "Actual on-time flight predicted as delayed",
        "FN": "Actual delayed flight predicted as on time",
        "TP": "Actual delayed flight predicted as delayed",
    }
    return pd.DataFrame(
        [
            {"CONFUSION_TERM": term, "MEANING": meanings[term], "COUNT": count}
            for term, count in [
                ("TN", int(tn)),
                ("FP", int(fp)),
                ("FN", int(fn)),
                ("TP", int(tp)),
            ]
        ]
    )


print("Shared preprocessing, threshold, and evaluation utilities created.")


## Class-Imbalance Strategy

Class imbalance is treated as a hyperparameter rather than being fixed in advance. Every algorithm compares:

- `natural`: the original training distribution without weighting;
- `class_weight`: estimator-specific minority-class weighting;
- `undersample`: reproducible random undersampling applied only to training observations.

Validation observations always retain their natural distribution. A common out-of-fold F2 threshold search makes Precision, Recall, and F-scores comparable even though weighting and sampling can shift predicted probabilities.


## Metric Definitions and Selection Roles

`ARR_DEL15 = 1` represents a delayed flight. Accuracy is reported only as a descriptive measure because the target is imbalanced.

- `DELAY_PRECISION = TP / (TP + FP)`: proportion of delay alerts that were correct.
- `DELAY_RECALL = TP / (TP + FN)`: proportion of actual delays detected.
- `DELAY_F1`: equal balance between delayed-flight Precision and Recall.
- `DELAY_F2`: weighted balance that emphasizes Recall twice as strongly as Precision.
- `PR_AUC`: minority-class ranking quality across thresholds; the primary tuning metric.
- `ROC_AUC`: overall class discrimination across thresholds.
- `BRIER_SCORE`: probability error; lower is better.
- `TOP_10_RECALL`: proportion of all delays captured in the highest-risk 10% of flights.
- `TOP_10_LIFT`: delay concentration in that group relative to the overall delay rate.

Broad-search progression is led by mean PR-AUC. Final selection is rule-based: candidates must be competitive under the one-standard-error PR-AUC rule, avoid outlier temporal instability, and meet a predeclared delayed-flight Recall guardrail. Eligible candidates are then compared lexicographically using delayed-flight F2 and F1, calibration, Top-10% usefulness, interpretability, and cost. No arbitrary weighted score determines the winner.


## Shared Baseline Dataset

Before hyperparameter tuning, the three standard-Python algorithms are trained and evaluated using the same bounded chronological datasets. Training observations come only from the January–August development period. Validation observations come from the later September–October validation period.

The same feature matrix and labels are reused across all three algorithms, ensuring that baseline differences arise from the algorithms rather than from different samples.


In [0]:
baseline_training_df = bounded_uniform_sample(
    df_train_prepared.select(*MODEL_INPUT_COLUMNS, TARGET_COLUMN),
    LOCAL_TRAIN_MAX_ROWS,
    seed=RANDOM_SEED,
)
baseline_validation_df = bounded_uniform_sample(
    df_validation_prepared.select(*MODEL_INPUT_COLUMNS, TARGET_COLUMN),
    LOCAL_VALIDATION_MAX_ROWS,
    seed=RANDOM_SEED + 1,
)

(
    baseline_preprocessor,
    X_baseline_train,
    y_baseline_train,
    X_baseline_validation,
    y_baseline_validation,
) = prepare_training_validation_matrices(
    baseline_training_df,
    baseline_validation_df,
)

print(f"Shared baseline training rows: {X_baseline_train.shape[0]:,}")
print(
    "Shared baseline validation rows: "
    f"{X_baseline_validation.shape[0]:,}"
)
print(
    "Baseline training delayed-flight rate: "
    f"{np.mean(y_baseline_train):.4f}"
)
print(
    "Baseline validation delayed-flight rate: "
    f"{np.mean(y_baseline_validation):.4f}"
)


## Train and Evaluate Standard-Python Baselines

The majority-class baseline demonstrates why Accuracy is insufficient. Untuned Logistic Regression, Random Forest, and XGBoost baselines use identical chronological matrices. These initial models are diagnostic references; imbalance strategy and hyperparameters are selected only through the five-fold tuning workflow.


In [0]:
majority_class = int(np.bincount(y_baseline_train).argmax())
majority_predictions = np.full(
    y_baseline_validation.shape,
    majority_class,
    dtype=np.int8,
)

majority_weighted_precision, majority_weighted_recall, majority_weighted_f1, _ = (
    precision_recall_fscore_support(
        y_baseline_validation,
        majority_predictions,
        average="weighted",
        zero_division=0,
    )
)

baseline_rows = [
    {
        "MODEL": "Majority Class Baseline",
        "DECISION_THRESHOLD": 0.5,
        "ACCURACY": float(
            accuracy_score(y_baseline_validation, majority_predictions)
        ),
        "PRECISION": float(majority_weighted_precision),
        "RECALL": float(majority_weighted_recall),
        "F1_SCORE": float(majority_weighted_f1),
        "ROC_AUC": None,
        "PR_AUC": None,
        "BRIER_SCORE": float(
            brier_score_loss(
                y_baseline_validation,
                majority_predictions.astype(float),
            )
        ),
        "TOP_10_RECALL": 0.0,
        "TOP_10_LIFT": 0.0,
        "DELAY_PRECISION": 0.0,
        "DELAY_RECALL": 0.0,
        "DELAY_F1": 0.0,
        "DELAY_F2": 0.0,
    }
]

baseline_estimators = {
    "Logistic Regression (Baseline)": LogisticRegression(
        C=1.0,
        penalty="l2",
        solver="lbfgs",
        max_iter=500,
        tol=1e-3,
        random_state=RANDOM_SEED,
    ),
    "Random Forest (Baseline)": RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ),
    "XGBoost (Baseline)": XGBClassifier(
        n_estimators=250,
        max_depth=6,
        learning_rate=0.05,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.0,
        reg_lambda=2.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ),
}

baseline_predictions = {}
for model_name, estimator in baseline_estimators.items():
    estimator.fit(X_baseline_train, y_baseline_train)
    probabilities = estimator.predict_proba(X_baseline_validation)[:, 1]
    predictions = (probabilities >= 0.5).astype(np.int8)
    baseline_predictions[model_name] = predictions
    baseline_rows.append(
        {
            "MODEL": model_name,
            "DECISION_THRESHOLD": 0.5,
            **evaluate_probabilities(
                y_baseline_validation,
                probabilities,
                threshold=0.5,
            ),
        }
    )

model_comparison = spark.createDataFrame(baseline_rows)
display(model_comparison.orderBy("MODEL"))


## Baseline Confusion Matrices

The confusion matrices below show how each untuned algorithm classified the same chronological validation observations. Rows represent actual outcomes and columns represent predicted outcomes.

- The upper-left cell is the number of correctly identified on-time flights (true negatives).
- The upper-right cell is the number of on-time flights incorrectly flagged as delayed (false positives).
- The lower-left cell is the number of delayed flights missed by the model (false negatives).
- The lower-right cell is the number of delayed flights correctly identified (true positives).

For this project, the lower-left cell is especially important because it contains actual delays that would receive no operational warning.


In [0]:
for model_name, predictions in baseline_predictions.items():
    print(f"Baseline confusion matrix: {model_name}")
    display(
        confusion_matrix_table(
            y_baseline_validation,
            predictions,
        )
    )
    print(f"Baseline TN/FP/FN/TP counts: {model_name}")
    display(
        confusion_count_table(
            y_baseline_validation,
            predictions,
        )
    )


## Baseline Comparison Interpretation

The majority-class baseline is diagnostic rather than operational: it can achieve high Accuracy while detecting no delayed flights. The three machine-learning baselines use the same unmodified training distribution and threshold `0.5`, providing a neutral pre-tuning reference.

No candidate is selected from this single September–October comparison. Final candidate selection is based exclusively on the five expanding chronological folds and the deep search defined below.


## Unified Chronological Cross-Validation

One cross-validation workflow is used for Logistic Regression, Random Forest, and XGBoost.

The January–August development period is divided into five expanding-window folds:

| Fold | Training period | Validation period |
|---|---|---|
| 1 | January–March 2025 | April 2025 |
| 2 | January–April 2025 | May 2025 |
| 3 | January–May 2025 | June 2025 |
| 4 | January–June 2025 | July 2025 |
| 5 | January–July 2025 | August 2025 |

For each fold, the chronological date boundaries are applied before sampling. One reproducible uniform training sample and one reproducible uniform validation sample are then created and converted to local sparse matrices. These exact matrices are reused by every algorithm and every hyperparameter configuration.

This arrangement prevents future observations from entering earlier training periods, keeps validation distributions natural, and avoids algorithm-specific sampling differences.


In [0]:
TUNING_FOLDS = [
    {
        "train_end": "2025-03-31",
        "validation_start": "2025-04-01",
        "validation_end": "2025-04-30",
    },
    {
        "train_end": "2025-04-30",
        "validation_start": "2025-05-01",
        "validation_end": "2025-05-31",
    },
    {
        "train_end": "2025-05-31",
        "validation_start": "2025-06-01",
        "validation_end": "2025-06-30",
    },
    {
        "train_end": "2025-06-30",
        "validation_start": "2025-07-01",
        "validation_end": "2025-07-31",
    },
    {
        "train_end": "2025-07-31",
        "validation_start": "2025-08-01",
        "validation_end": "2025-08-31",
    },
]

LOCAL_CV_FOLDS = []

for fold_number, fold in enumerate(TUNING_FOLDS, start=1):
    fold_training_raw = df_train.filter(
        F.col("FL_DATE") <= F.to_date(F.lit(fold["train_end"]))
    )
    fold_validation_raw = df_train.filter(
        F.col("FL_DATE").between(
            F.to_date(F.lit(fold["validation_start"])),
            F.to_date(F.lit(fold["validation_end"])),
        )
    )

    # Training rows retain causal day-by-day histories. Validation rows
    # receive mappings frozen at the fold's training cutoff.
    fold_global_rate, fold_history_maps = build_frozen_history_maps(
        fold_training_raw
    )
    fold_validation_hist = attach_frozen_history(
        fold_validation_raw,
        fold_global_rate,
        fold_history_maps,
    )

    complete_training_fold = df_train_prepared.filter(
        F.col("FL_DATE") <= F.to_date(F.lit(fold["train_end"]))
    )
    complete_validation_fold = (
        prepare_hist_modeling_frame(fold_validation_hist)
        .select("FL_DATE", *MODEL_INPUT_COLUMNS, TARGET_COLUMN)
    )

    sampled_training_fold = bounded_uniform_sample(
        complete_training_fold,
        LOCAL_TRAIN_MAX_ROWS,
        seed=RANDOM_SEED + fold_number * 10,
    )
    sampled_validation_fold = bounded_uniform_sample(
        complete_validation_fold,
        LOCAL_VALIDATION_MAX_ROWS,
        seed=RANDOM_SEED + fold_number * 10 + 1,
    )

    (
        fold_preprocessor,
        X_train_fold,
        y_train_fold,
        X_validation_fold,
        y_validation_fold,
    ) = prepare_training_validation_matrices(
        sampled_training_fold,
        sampled_validation_fold,
    )

    LOCAL_CV_FOLDS.append(
        {
            "fold": fold_number,
            "X_train": X_train_fold,
            "y_train": y_train_fold,
            "X_validation": X_validation_fold,
            "y_validation": y_validation_fold,
            "preprocessor": fold_preprocessor,
            "scale_pos_weight": positive_class_weight(y_train_fold),
        }
    )

    print(
        f"Fold {fold_number}: "
        f"{X_train_fold.shape[0]:,} training rows, "
        f"{X_validation_fold.shape[0]:,} validation rows, "
        f"training delay rate={np.mean(y_train_fold):.4f}, "
        f"validation delay rate={np.mean(y_validation_fold):.4f}"
    )

BROAD_CV_FOLDS = build_reduced_cv_folds(
    LOCAL_CV_FOLDS,
    training_maximum=BROAD_TRAIN_MAX_ROWS,
    validation_maximum=BROAD_VALIDATION_MAX_ROWS,
    seed_offset=1_000,
)

print("Five fold-isolated chronological matrices prepared.")
print(
    "Broad screening per fold: "
    f"{BROAD_CV_FOLDS[0]['X_train'].shape[0]:,} train / "
    f"{BROAD_CV_FOLDS[0]['X_validation'].shape[0]:,} validation"
)
print(
    "Confirmation per fold: "
    f"{LOCAL_CV_FOLDS[0]['X_train'].shape[0]:,} train / "
    f"{LOCAL_CV_FOLDS[0]['X_validation'].shape[0]:,} validation"
)


## Efficient Deep Two-Stage Hyperparameter Search

The first stage is a reproducible randomized broad screening over regularization, model complexity, learning dynamics, and all three imbalance strategies. It uses smaller but identically stratified chronological matrices for every algorithm.

The second stage takes the three strongest broad configurations per algorithm, adds unique local refinements around the broad winner, and evaluates that combined shortlist on the larger common confirmation folds. Final model selection uses confirmation results only, so no model is favored by being evaluated on a smaller sample.

Planned broad screening:

- 20 Logistic Regression configurations;
- 30 Random Forest configurations;
- 50 XGBoost configurations.

Planned local refinements:

- 8 Logistic Regression configurations;
- 10 Random Forest configurations;
- 15 XGBoost configurations.

With five folds, this produces at most approximately 710 fits. This is a deep randomized search followed by focused confirmation, not an impractical exhaustive Cartesian grid. Logistic Regression uses compatible solvers, bounded iterations, and a practical convergence tolerance; non-converged configurations remain in the audit output but cannot be selected.


In [0]:
def sampled_grid(distributions, *, n_iter, seed):
    '''Return a reproducible list of unique sampled configurations.'''
    maximum_combinations = int(
        np.prod([len(values) for values in distributions.values()])
    )
    sampled = ParameterSampler(
        distributions,
        n_iter=min(n_iter, maximum_combinations),
        random_state=seed,
    )
    unique = []
    seen = set()
    for configuration in sampled:
        configuration = dict(configuration)
        key = tuple(sorted(configuration.items()))
        if key not in seen:
            seen.add(key)
            unique.append(configuration)
    return unique


LOGISTIC_REGRESSION_BROAD_GRID = [
    *sampled_grid(
        {
            "C": [0.001, 0.005, 0.01, 0.05, 0.10, 0.50, 1.0],
            "penalty": ["l2"],
            "solver": ["lbfgs"],
            "imbalance_strategy": ["natural", "class_weight", "undersample"],
        },
        n_iter=7,
        seed=RANDOM_SEED + 101,
    ),
    *sampled_grid(
        {
            "C": [0.001, 0.005, 0.01, 0.05, 0.10, 0.50, 1.0],
            "penalty": ["l1"],
            "solver": ["saga"],
            "imbalance_strategy": ["natural", "class_weight", "undersample"],
        },
        n_iter=7,
        seed=RANDOM_SEED + 102,
    ),
    *sampled_grid(
        {
            "C": [0.001, 0.005, 0.01, 0.05, 0.10, 0.50, 1.0],
            "penalty": ["elasticnet"],
            "solver": ["saga"],
            "l1_ratio": [0.25, 0.50, 0.75],
            "imbalance_strategy": ["natural", "class_weight", "undersample"],
        },
        n_iter=6,
        seed=RANDOM_SEED + 103,
    ),
]

RANDOM_FOREST_BROAD_GRID = sampled_grid(
    {
        "n_estimators": [100, 200, 300, 400],
        "max_depth": [8, 12, 16, 24],
        "min_samples_split": [2, 5, 10, 20],
        "min_samples_leaf": [1, 2, 5, 10, 20],
        "max_features": ["sqrt", "log2", 0.25],
        "criterion": ["gini", "entropy", "log_loss"],
        "bootstrap": [True, False],
        "imbalance_strategy": ["natural", "class_weight", "undersample"],
    },
    n_iter=30,
    seed=RANDOM_SEED + 202,
)

XGBOOST_BROAD_GRID = sampled_grid(
    {
        "n_estimators": [100, 200, 300, 500],
        "max_depth": [3, 4, 6, 8, 10],
        "learning_rate": [0.01, 0.03, 0.05, 0.08, 0.12],
        "min_child_weight": [1, 3, 5, 10, 15],
        "subsample": [0.70, 0.85, 1.00],
        "colsample_bytree": [0.70, 0.85, 1.00],
        "gamma": [0.0, 0.10, 0.50, 1.0, 2.0],
        "reg_alpha": [0.0, 0.01, 0.10, 1.0, 5.0],
        "reg_lambda": [1.0, 2.0, 5.0, 10.0, 20.0],
        "imbalance_strategy": ["natural", "class_weight", "undersample"],
    },
    n_iter=50,
    seed=RANDOM_SEED + 303,
)

BROAD_SEARCH_FITS = (
    len(LOGISTIC_REGRESSION_BROAD_GRID)
    + len(RANDOM_FOREST_BROAD_GRID)
    + len(XGBOOST_BROAD_GRID)
) * len(TUNING_FOLDS)

print(
    "Broad configurations: "
    f"LR={len(LOGISTIC_REGRESSION_BROAD_GRID)}, "
    f"RF={len(RANDOM_FOREST_BROAD_GRID)}, "
    f"XGBoost={len(XGBOOST_BROAD_GRID)}"
)
print(f"Broad-screening chronological fits: {BROAD_SEARCH_FITS}")


## Estimator, Imbalance, and Refinement Builders

Builders create fresh estimators for every fold. Class weighting is activated only when selected by the configuration, and undersampling is performed reproducibly on training matrices only. Refinement spaces are generated around each broad-search winner and deduplicated before evaluation.


In [0]:
def build_logistic_regression(params, *, scale_pos_weight=None):
    params = dict(params)
    imbalance_strategy = params.pop("imbalance_strategy")
    return LogisticRegression(
        **params,
        class_weight=(
            "balanced" if imbalance_strategy == "class_weight" else None
        ),
        max_iter=500,
        tol=1e-3,
        random_state=RANDOM_SEED,
    )


def build_random_forest(params, *, scale_pos_weight=None):
    params = dict(params)
    imbalance_strategy = params.pop("imbalance_strategy")
    return RandomForestClassifier(
        **params,
        class_weight=(
            "balanced_subsample"
            if imbalance_strategy == "class_weight"
            else None
        ),
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )


def build_xgboost(params, *, scale_pos_weight):
    params = dict(params)
    imbalance_strategy = params.pop("imbalance_strategy")
    return XGBClassifier(
        **params,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        scale_pos_weight=(
            scale_pos_weight
            if imbalance_strategy == "class_weight"
            else 1.0
        ),
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )


def training_data_for_strategy(X_train, y_train, strategy, *, seed):
    '''Apply one imbalance treatment to training data only.'''
    if strategy in {"natural", "class_weight"}:
        return X_train, y_train
    if strategy != "undersample":
        raise ValueError(f"Unknown imbalance strategy: {strategy}")

    positive_indices = np.flatnonzero(y_train == 1)
    negative_indices = np.flatnonzero(y_train == 0)
    if len(positive_indices) == 0 or len(negative_indices) == 0:
        raise ValueError("Both classes are required for undersampling.")

    generator = np.random.default_rng(seed)
    retained_negative_indices = generator.choice(
        negative_indices,
        size=min(len(negative_indices), len(positive_indices)),
        replace=False,
    )
    retained_indices = np.concatenate(
        [positive_indices, retained_negative_indices]
    )
    generator.shuffle(retained_indices)
    return X_train[retained_indices], y_train[retained_indices]


def local_numeric_values(value, multipliers, *, minimum, maximum=None, integer=False):
    values = []
    for multiplier in multipliers:
        candidate = value * multiplier
        candidate = max(minimum, candidate)
        if maximum is not None:
            candidate = min(maximum, candidate)
        if integer:
            candidate = int(round(candidate))
        values.append(candidate)
    return sorted(set(values))


def focused_refinement_grid(
    model_name,
    best_params,
    *,
    excluded_configurations=None,
):
    '''Create unique local candidates around one broad-search winner.'''
    best_params = dict(best_params)
    excluded_keys = {tuple(sorted(best_params.items()))}
    for configuration in excluded_configurations or []:
        excluded_keys.add(tuple(sorted(dict(configuration).items())))

    if model_name == "Logistic Regression":
        distributions = {
            "C": local_numeric_values(
                best_params["C"],
                [0.40, 0.65, 1.0, 1.50, 2.50],
                minimum=1e-5,
            ),
            "penalty": [best_params["penalty"]],
            "solver": [best_params["solver"]],
            "imbalance_strategy": ["natural", "class_weight", "undersample"],
        }
        if best_params["penalty"] == "elasticnet":
            ratio = best_params["l1_ratio"]
            distributions["l1_ratio"] = sorted(
                set([max(0.05, ratio - 0.20), ratio, min(0.95, ratio + 0.20)])
            )
        target_size, seed = 8, RANDOM_SEED + 401

    elif model_name == "Random Forest":
        best_depth = best_params["max_depth"] or 24
        distributions = {
            "n_estimators": local_numeric_values(
                best_params["n_estimators"],
                [0.75, 1.0, 1.25],
                minimum=100,
                maximum=500,
                integer=True,
            ),
            "max_depth": local_numeric_values(
                best_depth,
                [0.75, 1.0, 1.25],
                minimum=4,
                maximum=32,
                integer=True,
            ),
            "min_samples_split": local_numeric_values(
                best_params["min_samples_split"],
                [0.50, 1.0, 2.0],
                minimum=2,
                maximum=40,
                integer=True,
            ),
            "min_samples_leaf": local_numeric_values(
                best_params["min_samples_leaf"],
                [0.50, 1.0, 2.0],
                minimum=1,
                maximum=40,
                integer=True,
            ),
            "max_features": [best_params["max_features"], "sqrt", "log2"],
            "criterion": [best_params["criterion"]],
            "bootstrap": [best_params["bootstrap"]],
            "imbalance_strategy": ["natural", "class_weight", "undersample"],
        }
        target_size, seed = 10, RANDOM_SEED + 402

    elif model_name == "XGBoost":
        distributions = {
            "n_estimators": local_numeric_values(
                best_params["n_estimators"],
                [0.75, 1.0, 1.25],
                minimum=100,
                maximum=500,
                integer=True,
            ),
            "max_depth": local_numeric_values(
                best_params["max_depth"],
                [0.75, 1.0, 1.25],
                minimum=3,
                maximum=12,
                integer=True,
            ),
            "learning_rate": local_numeric_values(
                best_params["learning_rate"],
                [0.70, 1.0, 1.30],
                minimum=0.005,
                maximum=0.20,
            ),
            "min_child_weight": local_numeric_values(
                best_params["min_child_weight"],
                [0.50, 1.0, 2.0],
                minimum=1,
                maximum=30,
                integer=True,
            ),
            "subsample": sorted(set([best_params["subsample"], 0.80, 0.90, 1.0])),
            "colsample_bytree": sorted(
                set([best_params["colsample_bytree"], 0.80, 0.90, 1.0])
            ),
            "gamma": sorted(set([best_params["gamma"], 0.0, 0.50, 1.0])),
            "reg_alpha": sorted(set([best_params["reg_alpha"], 0.0, 0.10, 1.0])),
            "reg_lambda": local_numeric_values(
                best_params["reg_lambda"],
                [0.50, 1.0, 2.0],
                minimum=0.10,
                maximum=40.0,
            ),
            "imbalance_strategy": ["natural", "class_weight", "undersample"],
        }
        target_size, seed = 15, RANDOM_SEED + 403

    else:
        raise ValueError(f"Unsupported model for refinement: {model_name}")

    sampled = sampled_grid(
        distributions,
        n_iter=target_size + 15,
        seed=seed,
    )
    return [
        params
        for params in sampled
        if tuple(sorted(params.items())) not in excluded_keys
    ][:target_size]


print("Estimator, imbalance, and refinement builders created.")


## Unified Five-Fold Screening and Confirmation

Every broad candidate is fitted on the same five reduced chronological folds. Mean PR-AUC selects the broad leader and identifies the top three configurations for each algorithm.

Those top configurations and unique nearby refinements are then refitted on the same five larger confirmation folds. Probability metrics are calculated independently by fold. A single operating threshold is selected from pooled out-of-fold probabilities by maximizing delayed-flight F2 and is applied back to every fold for comparable mean and stability statistics.

Only converged confirmation-stage configurations are eligible for final rule-based selection. Broad-screening results remain available as tuning evidence but never compete directly against confirmation results.


In [0]:
RESULT_COLUMNS = [
    "MODEL",
    "SEARCH_STAGE",
    "PARAMETERS",
    "CONVERGED",
    "DECISION_THRESHOLD",
    "ACCURACY",
    "PRECISION",
    "RECALL",
    "F1_SCORE",
    "ROC_AUC",
    "PR_AUC",
    "BRIER_SCORE",
    "TOP_10_RECALL",
    "TOP_10_LIFT",
    "DELAY_PRECISION",
    "DELAY_RECALL",
    "DELAY_F1",
    "DELAY_F2",
    "TRAINING_SECONDS",
    "PR_AUC_STD",
    "DELAY_RECALL_STD",
    "DELAY_F1_STD",
    "DELAY_F2_STD",
    "BRIER_SCORE_STD",
]


def tune_python_classifier(
    model_name,
    estimator_builder,
    parameter_grid,
    *,
    search_stage,
    local_folds=LOCAL_CV_FOLDS,
):
    '''Evaluate one candidate family on shared chronological folds.'''
    if not parameter_grid:
        raise ValueError(f"{model_name} parameter grid is empty.")

    result_rows = []
    for configuration_number, params in enumerate(parameter_grid, start=1):
        fold_outputs = []
        converged = True

        for fold in local_folds:
            X_fit, y_fit = training_data_for_strategy(
                fold["X_train"],
                fold["y_train"],
                params["imbalance_strategy"],
                seed=(
                    RANDOM_SEED
                    + fold["fold"] * 100
                    + configuration_number
                ),
            )
            estimator = estimator_builder(
                params,
                scale_pos_weight=fold["scale_pos_weight"],
            )

            training_start = time.perf_counter()
            with warnings.catch_warnings(record=True) as captured_warnings:
                warnings.simplefilter("always", ConvergenceWarning)
                estimator.fit(X_fit, y_fit)
            training_seconds = time.perf_counter() - training_start

            if any(
                issubclass(item.category, ConvergenceWarning)
                for item in captured_warnings
            ):
                converged = False

            fold_outputs.append(
                {
                    "labels": fold["y_validation"],
                    "probabilities": estimator.predict_proba(
                        fold["X_validation"]
                    )[:, 1],
                    "training_seconds": float(training_seconds),
                }
            )

        pooled_labels = np.concatenate(
            [item["labels"] for item in fold_outputs]
        )
        pooled_probabilities = np.concatenate(
            [item["probabilities"] for item in fold_outputs]
        )
        decision_threshold = select_operating_threshold(
            pooled_labels,
            pooled_probabilities,
        )

        fold_metrics = []
        for item in fold_outputs:
            metrics = evaluate_probabilities(
                item["labels"],
                item["probabilities"],
                threshold=decision_threshold,
            )
            metrics["TRAINING_SECONDS"] = item["training_seconds"]
            fold_metrics.append(metrics)

        mean_metric_names = [
            "ACCURACY",
            "PRECISION",
            "RECALL",
            "F1_SCORE",
            "ROC_AUC",
            "PR_AUC",
            "BRIER_SCORE",
            "TOP_10_RECALL",
            "TOP_10_LIFT",
            "DELAY_PRECISION",
            "DELAY_RECALL",
            "DELAY_F1",
            "DELAY_F2",
            "TRAINING_SECONDS",
        ]
        averaged = {
            metric_name: round(
                float(np.mean([row[metric_name] for row in fold_metrics])),
                2 if metric_name == "TRAINING_SECONDS" else 4,
            )
            for metric_name in mean_metric_names
        }
        stability = {
            f"{metric_name}_STD": round(
                float(
                    np.std(
                        [row[metric_name] for row in fold_metrics],
                        ddof=0,
                    )
                ),
                4,
            )
            for metric_name in [
                "PR_AUC",
                "DELAY_RECALL",
                "DELAY_F1",
                "DELAY_F2",
                "BRIER_SCORE",
            ]
        }

        result_rows.append(
            {
                "MODEL": model_name,
                "SEARCH_STAGE": search_stage,
                "PARAMETERS": str(params),
                "CONVERGED": bool(converged),
                "DECISION_THRESHOLD": float(decision_threshold),
                **averaged,
                **stability,
            }
        )

        print(
            f"{model_name} {search_stage} "
            f"{configuration_number}/{len(parameter_grid)} | "
            f"PR AUC={averaged['PR_AUC']:.4f}, "
            f"delay F2={averaged['DELAY_F2']:.4f}, "
            f"threshold={decision_threshold:.2f}, "
            f"converged={converged}"
        )

    return spark.createDataFrame(result_rows).select(*RESULT_COLUMNS)


def rank_tuning_results(results):
    '''Rank configurations for broad-to-refinement progression.'''
    return results.orderBy(
        F.desc("CONVERGED"),
        F.desc("PR_AUC"),
        F.asc("PR_AUC_STD"),
        F.desc("DELAY_F2"),
        F.desc("DELAY_F1"),
        F.asc("BRIER_SCORE"),
        F.desc("TOP_10_LIFT"),
        F.asc("TRAINING_SECONDS"),
    )



def unique_parameter_configurations(configurations):
    '''Deduplicate parameter dictionaries while preserving order.'''
    unique = []
    seen = set()
    for configuration in configurations:
        configuration = dict(configuration)
        key = tuple(sorted(configuration.items()))
        if key not in seen:
            seen.add(key)
            unique.append(configuration)
    return unique


def run_two_stage_search(model_name, estimator_builder, broad_grid):
    '''Screen broadly, then confirm a focused shortlist on larger folds.'''
    broad_results = tune_python_classifier(
        model_name=model_name,
        estimator_builder=estimator_builder,
        parameter_grid=broad_grid,
        search_stage="broad_screening",
        local_folds=BROAD_CV_FOLDS,
    )

    ranked_converged_broad = rank_tuning_results(
        broad_results.filter(F.col("CONVERGED") == F.lit(True))
    )
    top_broad_rows = ranked_converged_broad.limit(3).collect()
    if not top_broad_rows:
        raise ValueError(
            f"No converged broad-search result exists for {model_name}."
        )

    top_broad_parameters = [
        ast.literal_eval(row["PARAMETERS"])
        for row in top_broad_rows
    ]
    focused_grid = focused_refinement_grid(
        model_name,
        top_broad_parameters[0],
        excluded_configurations=top_broad_parameters,
    )
    confirmation_grid = unique_parameter_configurations(
        [*top_broad_parameters, *focused_grid]
    )

    print(
        f"{model_name}: confirming {len(confirmation_grid)} "
        "top/refined configurations on the larger folds."
    )
    confirmation_results = tune_python_classifier(
        model_name=model_name,
        estimator_builder=estimator_builder,
        parameter_grid=confirmation_grid,
        search_stage="confirmation",
        local_folds=LOCAL_CV_FOLDS,
    )

    return rank_tuning_results(
        broad_results.unionByName(confirmation_results)
    )


print("Unified threshold-aware tuning function created.")


## Tune Logistic Regression

Logistic Regression uses a two-stage search across all five chronological folds. The broad stage compares regularization, penalty, solver, and class-imbalance strategies. A focused stage then explores nearby settings around the broad-stage leader. Mean PR AUC guides the search, with fold-to-fold stability and delayed-class metrics used as supporting evidence.


In [0]:
lr_tuning_results = run_two_stage_search(
    model_name="Logistic Regression",
    estimator_builder=build_logistic_regression,
    broad_grid=LOGISTIC_REGRESSION_BROAD_GRID,
)

display(lr_tuning_results)


## Tune Random Forest

Random Forest uses the same two-stage, five-fold workflow. The search compares tree count, depth, minimum leaf size, feature subsampling, split criterion, and class-imbalance strategy. Validation data retain their natural class distribution.


In [0]:
rf_tuning_results = run_two_stage_search(
    model_name="Random Forest",
    estimator_builder=build_random_forest,
    broad_grid=RANDOM_FOREST_BROAD_GRID,
)

display(rf_tuning_results)


## Tune XGBoost

XGBoost uses the same two-stage, five-fold workflow. The search compares boosting rounds, depth, learning rate, child-weight, row and feature subsampling, regularization, minimum loss reduction, and class-imbalance strategy. Any class weight is calculated only from the current training fold.


In [0]:
xgb_tuning_results = run_two_stage_search(
    model_name="XGBoost",
    estimator_builder=build_xgboost,
    broad_grid=XGBOOST_BROAD_GRID,
)

display(xgb_tuning_results)


## Rule-Based Final-Model Selection

Final selection is performed across every converged **confirmation-stage** configuration. Broad-screening metrics are retained for audit and refinement decisions but cannot compete directly because they were calculated on smaller samples.

Eligibility is determined before the final metrics are inspected:

1. **Competitive PR-AUC:** the configuration must fall within one standard error of the best confirmation PR-AUC.
2. **Temporal stability:** its fold-to-fold PR-AUC standard deviation cannot be an upper-outlier under Tukey's `Q3 + 1.5 × IQR` rule.
3. **Operational detection:** delayed-flight Recall must be at least `0.60`. This is a predeclared project guardrail requiring the model to detect a clear majority of delays; it is not estimated from the holdout test.

Eligible configurations are ranked lexicographically by delayed-flight F2, delayed-flight F1, Brier Score, Top-10% Recall and Lift, PR-AUC, interpretability, and training time. This explicit sequence replaces the former manually weighted selection score.

Three secondary priority profiles—balanced, detection, and probability—report sensitivity without changing the primary selection rule.


In [0]:
complete_tuning_results = (
    lr_tuning_results
    .unionByName(rf_tuning_results)
    .unionByName(xgb_tuning_results)
)
all_tuning_results = (
    complete_tuning_results
    .filter(F.col("CONVERGED") == F.lit(True))
    .filter(F.col("SEARCH_STAGE") == F.lit("confirmation"))
)
candidate_rows = [row.asDict() for row in all_tuning_results.collect()]
if not candidate_rows:
    raise ValueError("No converged confirmation configurations are available.")


# One-standard-error competitiveness boundary around the best PR-AUC result.
best_pr_auc_row = max(
    candidate_rows,
    key=lambda row: (float(row["PR_AUC"]), -float(row["PR_AUC_STD"])),
)
BEST_CONFIRMATION_PR_AUC = float(best_pr_auc_row["PR_AUC"])
BEST_PR_AUC_STANDARD_ERROR = float(
    best_pr_auc_row["PR_AUC_STD"] / np.sqrt(len(TUNING_FOLDS))
)
PR_AUC_ELIGIBILITY_FLOOR = (
    BEST_CONFIRMATION_PR_AUC - BEST_PR_AUC_STANDARD_ERROR
)

# Data-driven stability boundary: only upper-outlier variability is rejected.
pr_auc_stability = np.asarray(
    [float(row["PR_AUC_STD"]) for row in candidate_rows],
    dtype=float,
)
stability_q1, stability_q3 = np.quantile(pr_auc_stability, [0.25, 0.75])
stability_iqr = float(stability_q3 - stability_q1)
PR_AUC_STABILITY_UPPER_FENCE = float(
    stability_q3 + 1.5 * stability_iqr
)

MINIMUM_DELAY_RECALL = 0.60
INTERPRETABILITY_RANKS = {
    "Logistic Regression": 1,
    "Random Forest": 2,
    "XGBoost": 3,
}

for row in candidate_rows:
    row["PR_AUC_GAP_FROM_BEST"] = round(
        BEST_CONFIRMATION_PR_AUC - float(row["PR_AUC"]),
        6,
    )
    row["PR_AUC_COMPETITIVE"] = bool(
        float(row["PR_AUC"]) >= PR_AUC_ELIGIBILITY_FLOOR
    )
    row["TEMPORALLY_STABLE"] = bool(
        float(row["PR_AUC_STD"]) <= PR_AUC_STABILITY_UPPER_FENCE
    )
    row["MEETS_RECALL_REQUIREMENT"] = bool(
        float(row["DELAY_RECALL"]) >= MINIMUM_DELAY_RECALL
    )
    row["INTERPRETABILITY_RANK"] = int(
        INTERPRETABILITY_RANKS[row["MODEL"]]
    )
    row["ELIGIBLE"] = bool(
        row["PR_AUC_COMPETITIVE"]
        and row["TEMPORALLY_STABLE"]
        and row["MEETS_RECALL_REQUIREMENT"]
    )

eligible_rows = [row for row in candidate_rows if row["ELIGIBLE"]]
if not eligible_rows:
    raise ValueError(
        "No confirmation configuration satisfies the predeclared PR-AUC, "
        "stability, and delayed-flight Recall requirements. Review the "
        "guardrails before inspecting holdout-test outcomes."
    )


SELECTION_ORDER = [
    F.desc("DELAY_F2"),
    F.desc("DELAY_F1"),
    F.asc("BRIER_SCORE"),
    F.desc("TOP_10_RECALL"),
    F.desc("TOP_10_LIFT"),
    F.desc("PR_AUC"),
    F.asc("INTERPRETABILITY_RANK"),
    F.asc("TRAINING_SECONDS"),
]

ranked_confirmation_candidates = (
    spark.createDataFrame(candidate_rows)
    .orderBy(
        F.desc("ELIGIBLE"),
        *SELECTION_ORDER,
    )
)
ranked_model_candidates = (
    ranked_confirmation_candidates
    .filter(F.col("ELIGIBLE") == F.lit(True))
    .orderBy(*SELECTION_ORDER)
)


def best_algorithm_candidate(model_name):
    '''Return the strongest available configuration for one algorithm.'''
    return (
        ranked_confirmation_candidates
        .filter(F.col("MODEL") == model_name)
        .limit(1)
    )


best_logistic_regression = best_algorithm_candidate("Logistic Regression")
best_random_forest = best_algorithm_candidate("Random Forest")
best_xgboost = best_algorithm_candidate("XGBoost")
tuned_model_comparison = (
    best_logistic_regression
    .unionByName(best_random_forest)
    .unionByName(best_xgboost)
    .orderBy(F.desc("ELIGIBLE"), *SELECTION_ORDER)
)


SELECTION_PRIORITY_PROFILES = {
    "BALANCED": [
        "DELAY_F2",
        "DELAY_F1",
        "PR_AUC",
        "BRIER_SCORE",
        "TOP_10_LIFT",
        "TRAINING_SECONDS",
    ],
    "DETECTION": [
        "DELAY_RECALL",
        "DELAY_F2",
        "DELAY_PRECISION",
        "DELAY_F1",
        "PR_AUC",
    ],
    "PROBABILITY": [
        "PR_AUC",
        "BRIER_SCORE",
        "ROC_AUC",
        "PR_AUC_STD",
        "TOP_10_LIFT",
    ],
}


def profile_selection_key(row, profile_name):
    '''Return a lexicographic sensitivity key without numerical weights.'''
    if profile_name == "BALANCED":
        return (
            row["DELAY_F2"],
            row["DELAY_F1"],
            row["PR_AUC"],
            -row["BRIER_SCORE"],
            row["TOP_10_LIFT"],
            -row["TRAINING_SECONDS"],
        )
    if profile_name == "DETECTION":
        return (
            row["DELAY_RECALL"],
            row["DELAY_F2"],
            row["DELAY_PRECISION"],
            row["DELAY_F1"],
            row["PR_AUC"],
        )
    if profile_name == "PROBABILITY":
        return (
            row["PR_AUC"],
            -row["BRIER_SCORE"],
            row["ROC_AUC"],
            -row["PR_AUC_STD"],
            row["TOP_10_LIFT"],
        )
    raise ValueError(f"Unknown selection profile: {profile_name}")


primary_winner = max(
    eligible_rows,
    key=lambda row: (
        row["DELAY_F2"],
        row["DELAY_F1"],
        -row["BRIER_SCORE"],
        row["TOP_10_RECALL"],
        row["TOP_10_LIFT"],
        row["PR_AUC"],
        -row["INTERPRETABILITY_RANK"],
        -row["TRAINING_SECONDS"],
    ),
)
sensitivity_rows = [
    {
        "SELECTION_PROFILE": "PRIMARY_RULE",
        "MODEL": primary_winner["MODEL"],
        "PARAMETERS": primary_winner["PARAMETERS"],
        "DECISION_THRESHOLD": primary_winner["DECISION_THRESHOLD"],
        "PRIMARY_METRIC": "DELAY_F2",
        "PRIMARY_VALUE": float(primary_winner["DELAY_F2"]),
    }
]

for profile_name in SELECTION_PRIORITY_PROFILES:
    winner = max(
        eligible_rows,
        key=lambda row: profile_selection_key(row, profile_name),
    )
    primary_metric = SELECTION_PRIORITY_PROFILES[profile_name][0]
    sensitivity_rows.append(
        {
            "SELECTION_PROFILE": profile_name,
            "MODEL": winner["MODEL"],
            "PARAMETERS": winner["PARAMETERS"],
            "DECISION_THRESHOLD": winner["DECISION_THRESHOLD"],
            "PRIMARY_METRIC": primary_metric,
            "PRIMARY_VALUE": float(winner[primary_metric]),
        }
    )

selection_sensitivity = spark.createDataFrame(sensitivity_rows)

print(f"Best confirmation PR AUC: {BEST_CONFIRMATION_PR_AUC:.4f}")
print(f"One-standard-error PR AUC floor: {PR_AUC_ELIGIBILITY_FLOOR:.4f}")
print(
    "PR AUC stability upper fence: "
    f"{PR_AUC_STABILITY_UPPER_FENCE:.4f}"
)
print(f"Minimum delayed-flight Recall: {MINIMUM_DELAY_RECALL:.2f}")
print(f"Eligible configurations: {len(eligible_rows)}/{len(candidate_rows)}")
print("Best available configuration from each algorithm:")
display(tuned_model_comparison)
print("Rule-based eligible configurations:")
display(ranked_model_candidates.limit(20))
print("Selection sensitivity without numerical weights:")
display(selection_sensitivity)
print("Complete confirmation-stage eligibility audit:")
display(ranked_confirmation_candidates)


## Tuned Confusion Matrices and Calibration Diagnostics

The strongest robust configuration from each algorithm is refitted independently on every fold. Its saved cross-validated threshold is applied to pooled out-of-fold probabilities for the confusion matrix, while calibration curves remain threshold-independent.

These are model-selection diagnostics on identical chronological observations. They make false positives, false negatives, calibration error, and the effect of each selected threshold directly visible.


In [0]:
best_parameter_rows = {
    "Logistic Regression": best_logistic_regression.first(),
    "Random Forest": best_random_forest.first(),
    "XGBoost": best_xgboost.first(),
}
estimator_builders = {
    "Logistic Regression": build_logistic_regression,
    "Random Forest": build_random_forest,
    "XGBoost": build_xgboost,
}

tuned_confusion_matrices = {}
tuned_probability_diagnostics = {}

for model_name, result_row in best_parameter_rows.items():
    if result_row is None:
        raise ValueError(f"No tuned result found for {model_name}.")

    best_parameters = ast.literal_eval(result_row["PARAMETERS"])
    decision_threshold = float(result_row["DECISION_THRESHOLD"])
    pooled_actual = []
    pooled_probability = []

    for fold in LOCAL_CV_FOLDS:
        estimator = estimator_builders[model_name](
            best_parameters,
            scale_pos_weight=fold["scale_pos_weight"],
        )
        fit_X, fit_y = training_data_for_strategy(
            fold["X_train"],
            fold["y_train"],
            best_parameters["imbalance_strategy"],
            seed=RANDOM_SEED + fold["fold"] * 100,
        )
        estimator.fit(fit_X, fit_y)
        pooled_actual.append(fold["y_validation"])
        pooled_probability.append(
            estimator.predict_proba(fold["X_validation"])[:, 1]
        )

    pooled_actual = np.concatenate(pooled_actual)
    pooled_probability = np.concatenate(pooled_probability)
    pooled_predicted = (
        pooled_probability >= decision_threshold
    ).astype(np.int8)

    tuned_confusion_matrices[model_name] = confusion_matrix_table(
        pooled_actual,
        pooled_predicted,
    )
    print(
        f"Tuned chronological confusion matrix: {model_name} "
        f"(threshold={decision_threshold:.2f})"
    )
    display(tuned_confusion_matrices[model_name])
    display(confusion_count_table(pooled_actual, pooled_predicted))

    probability_true, probability_predicted = calibration_curve(
        pooled_actual,
        pooled_probability,
        n_bins=10,
        strategy="quantile",
    )
    tuned_probability_diagnostics[model_name] = {
        "actual": pooled_actual,
        "probability": pooled_probability,
        "brier_score": brier_score_loss(
            pooled_actual,
            pooled_probability,
        ),
    }
    plt.plot(
        probability_predicted,
        probability_true,
        marker="o",
        label=(
            f"{model_name} "
            f"(Brier={tuned_probability_diagnostics[model_name]['brier_score']:.4f})"
        ),
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="black",
    label="Perfect calibration",
)
plt.xlabel("Mean predicted delay probability")
plt.ylabel("Observed delayed-flight rate")
plt.title("Chronological Cross-Validation Calibration Curves")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## Select the Candidate Final Model

The candidate is the first eligible configuration under the predeclared lexicographic rule. It has already passed the PR-AUC competitiveness, temporal-stability, and minimum delayed-flight Recall guardrails.

The sensitivity table must be discussed with the result. Agreement across balanced, detection, and probability priorities is strong evidence that the choice is robust; disagreement indicates a genuine operational trade-off that must be reported rather than hidden.


In [0]:
selected_model_row = ranked_model_candidates.first()
if selected_model_row is None:
    raise ValueError("The eligible model comparison contains no results.")

SELECTED_MODEL_NAME = selected_model_row["MODEL"]
SELECTED_MODEL_PARAMETERS = ast.literal_eval(
    selected_model_row["PARAMETERS"]
)
SELECTED_DECISION_THRESHOLD = float(
    selected_model_row["DECISION_THRESHOLD"]
)

selected_model_summary = ranked_model_candidates.limit(1)
display(selected_model_summary)
print(f"Selected model: {SELECTED_MODEL_NAME}")
print(f"Selected parameters: {SELECTED_MODEL_PARAMETERS}")
print(f"Cross-validated decision threshold: {SELECTED_DECISION_THRESHOLD:.2f}")
print(
    "Selection rule: one-standard-error PR-AUC eligibility, "
    "temporal-stability and Recall guardrails, followed by "
    "F2/F1, calibration, Top-10% usefulness, interpretability, and cost."
)


In [0]:
test

## Candidate-Selection Interpretation

Interpret the selected row only after the revised notebook finishes. Report mean and standard deviation across the five folds, the selected threshold, confusion counts, PR-AUC relative to delayed-flight prevalence, Brier Score, Top-10% Recall and Lift, training cost, and whether the three sensitivity profiles agree.

Also report the eligibility boundaries: the one-standard-error PR-AUC floor, the stability upper fence, and the `0.60` delayed-flight Recall guardrail. The selected row is the model-training candidate supported by the complete chronological tuning evidence. November–December outcomes were not used in preprocessing, imbalance treatment, hyperparameter refinement, threshold selection, guardrail construction, or model selection.


## Persist Modeling Checkpoints

This section saves the leakage-safe raw modeling tables, the complete confirmation-stage eligibility audit, the preprocessing specification, search design, eligibility guardrails, selected algorithm, imbalance strategy, hyperparameters, and cross-validated operating threshold.

Feature hashing is not used. The manifest records the standard-Python `ColumnTransformer` boundary and the exact package versions used by the executed notebook.


In [0]:
import json

checkpoint_tables = [
    (cfg.MODELING_TRAIN_HIST_TABLE, df_train_hist),
    (cfg.MODELING_VALIDATION_HIST_TABLE, df_validation_hist),
    (cfg.MODELING_TEST_HIST_TABLE, df_test_hist),
]

for table_name, dataframe in checkpoint_tables:
    row_count = dataframe.count()
    print(f"Saving {table_name}: {row_count:,} rows")
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

(
    ranked_confirmation_candidates.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(cfg.TUNED_MODEL_COMPARISON_TABLE)
)

feature_manifest = {
    "model_input_columns": MODEL_INPUT_COLUMNS,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "numerical_columns": NUMERICAL_COLUMNS,
    "target_column": TARGET_COLUMN,
    "preprocessing_library": "scikit-learn",
    "preprocessing_transformer": "ColumnTransformer",
    "categorical_transform": (
        "most-frequent imputation plus "
        "OneHotEncoder(handle_unknown='infrequent_if_exist', "
        "min_frequency=20)"
    ),
    "numerical_transform": (
        "median imputation plus StandardScaler(with_mean=False)"
    ),
    "fit_boundary": "fit on training only within each chronological fold",
    "historical_feature_boundary": (
        "validation mappings frozen at each fold training cutoff"
    ),
    "library_versions": LIBRARY_VERSIONS,
    "selected_model_name": SELECTED_MODEL_NAME,
    "selected_model_parameters": SELECTED_MODEL_PARAMETERS,
    "selected_decision_threshold": SELECTED_DECISION_THRESHOLD,
}

candidate_selection = {
    "selected_model_name": SELECTED_MODEL_NAME,
    "selected_model_parameters": SELECTED_MODEL_PARAMETERS,
    "selected_decision_threshold": SELECTED_DECISION_THRESHOLD,
    "selection_source": (
        "Five-fold randomized broad screening plus larger-fold "
        "confirmation and rule-based eligibility"
    ),
    "selected_after_hyperparameter_tuning": True,
    "modeling_implementation": "standard_python",
    "cross_validation": "five expanding chronological folds",
    "class_imbalance_alternatives": [
        "natural",
        "class_weight",
        "training_only_undersample",
    ],
    "threshold_policy": (
        "maximize delayed-flight F2 on pooled out-of-fold predictions"
    ),
    "selection_rule": (
        "one-standard-error PR-AUC eligibility; Tukey stability fence; "
        "minimum 0.60 delayed Recall; then lexicographic F2, F1, Brier, "
        "Top-10% usefulness, PR-AUC, interpretability, and cost"
    ),
    "eligibility_guardrails": {
        "best_confirmation_pr_auc": BEST_CONFIRMATION_PR_AUC,
        "best_pr_auc_standard_error": BEST_PR_AUC_STANDARD_ERROR,
        "pr_auc_floor": PR_AUC_ELIGIBILITY_FLOOR,
        "pr_auc_stability_upper_fence": PR_AUC_STABILITY_UPPER_FENCE,
        "minimum_delayed_recall": MINIMUM_DELAY_RECALL,
    },
    "selection_priority_profiles": SELECTION_PRIORITY_PROFILES,
    "confirmation_candidate_count": len(candidate_rows),
    "eligible_candidate_count": len(eligible_rows),
    "broad_search_sizes": {
        "Logistic Regression": len(LOGISTIC_REGRESSION_BROAD_GRID),
        "Random Forest": len(RANDOM_FOREST_BROAD_GRID),
        "XGBoost": len(XGBOOST_BROAD_GRID),
    },
    "confirmation_search_sizes": {
        "Logistic Regression": int(
            lr_tuning_results.filter(
                F.col("SEARCH_STAGE") == "confirmation"
            ).count()
        ),
        "Random Forest": int(
            rf_tuning_results.filter(
                F.col("SEARCH_STAGE") == "confirmation"
            ).count()
        ),
        "XGBoost": int(
            xgb_tuning_results.filter(
                F.col("SEARCH_STAGE") == "confirmation"
            ).count()
        ),
    },
    "tuning_sample_sizes_per_fold": {
        "broad_training": BROAD_TRAIN_MAX_ROWS,
        "broad_validation": BROAD_VALIDATION_MAX_ROWS,
        "confirmation_training": LOCAL_TRAIN_MAX_ROWS,
        "confirmation_validation": LOCAL_VALIDATION_MAX_ROWS,
    },
    "library_versions": LIBRARY_VERSIONS,
}

dbutils.fs.put(
    cfg.MODEL_FEATURE_MANIFEST_PATH,
    json.dumps(feature_manifest, indent=4),
    overwrite=True,
)
dbutils.fs.put(
    cfg.CANDIDATE_SELECTION_PATH,
    json.dumps(candidate_selection, indent=4),
    overwrite=True,
)

print("Model-training checkpoints saved successfully.")
print(f"Preprocessing manifest: {cfg.MODEL_FEATURE_MANIFEST_PATH}")
print(f"Candidate selection: {cfg.CANDIDATE_SELECTION_PATH}")


In [0]:
candidate_selection_saved = json.loads(
    dbutils.fs.head(cfg.CANDIDATE_SELECTION_PATH, 100_000)
)
print(json.dumps(candidate_selection_saved, indent=4))

assert candidate_selection_saved["selected_model_name"] == SELECTED_MODEL_NAME
assert (
    candidate_selection_saved["selected_model_parameters"]
    == SELECTED_MODEL_PARAMETERS
)
assert np.isclose(
    candidate_selection_saved["selected_decision_threshold"],
    SELECTED_DECISION_THRESHOLD,
)
print("Candidate-selection metadata verified.")
